<a href="https://colab.research.google.com/github/Iberasa3/BlackAndOchre/blob/main/Avispas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vespa velutina macro analysis notebook:

## This section is basic dataframe processing and analysis


In [ ]:
import pandas as pd

# The '\t' separator tells pandas to use the tab key as a delimiter
df = pd.read_csv('avispas_avistamientos.csv', sep='\t')
print(f"Number of columns: {df.shape[1]}")
print(f"Column names: {df.columns.tolist()[:5]}...")

In [ ]:
df['infraspecificEpithet'].unique() # Just cheking how many subespecies there are

In [ ]:
print(df['infraspecificEpithet'].value_counts(dropna=False)) #Checking instances of subespecies

In [ ]:
europe = ['ES', 'FR', 'PT', 'BE', 'IT', 'DE', 'CH', 'LU', 'GB', 'IE', 'NL']
# The analysis is focused on Europe, so we are keeping only European countries.
# Most of the records are European anyway.
df = df[df['countryCode'].isin(europe)]

print(f"Number of records changed from {len(df)} to {len(df)}.")

In [ ]:
# Counting sightings per country and sorting from highest to lowest
country_counts = df['countryCode'].value_counts()

print("Sightings by Country in Europe")
print(country_counts)

In [ ]:
# Cleaning coordinates: removing zero values and nulls
df = df[(df['decimalLatitude'] != 0.0) & (df['decimalLongitude'] != 0.0)]
df = df.dropna(subset=['decimalLatitude', 'decimalLongitude'])

print(f"Original records: {len(df)}")
print(f"Records after removing nulls and null-islands: {len(df)}")

In [ ]:
# Removing instances where uncertainty exceeds 1km.
max_threshold = 1000
df = df[(df['coordinateUncertaintyInMeters'] <= max_threshold) & (df['year'] >= 2005)]

In [ ]:
df = df[df['basisOfRecord'] != 'PRESERVED_SPECIMEN']
print(df['basisOfRecord'].unique())

## Spatial Thinning and Data Independence

We are implementing **spatial thinning** to address spatial autocorrelation. Since we are using a **Random Forest** model, we must satisfy the assumption of independence between observations. Clustered points often reflect sightings from the same colony rather than independent distribution events.

Given that *Vespa velutina* typically forages within a **1 to 1.5 km radius** of its nest, sightings within this range are likely linked to the same location. By applying spatial thinning at this scale, we ensure that:
1. We avoid **oversampling** in highly monitored areas.
2. We treat the project as a **distribution model** (presence/absence) rather than an abundance model.
3. The spatial resolution remains consistent with the species' biological range.



In [ ]:
# Checking for and removing duplicate records based on gbifID
num_duplicates = df.duplicated(subset='gbifID').sum()
print(f"Records with the same gbifID: {num_duplicates}")

# Keeping only the first occurrence of each unique ID
df.drop_duplicates(subset='gbifID', inplace=True)

In [ ]:
# 0.015 degrees of latitude is approximately 1.1 km.
# Each grid cell will be roughly 1.1 km x 1.1 km,
# aligning with the standard 1km resolution for bioclimatic models.
res = 0.015

# Divide by resolution, round to the nearest integer, and multiply back.
df['lat_grid'] = (df['decimalLatitude'] / res).round() * res
df['lon_grid'] = (df['decimalLongitude'] / res).round() * res

print(f"Original record count: {len(df)}")

# Dropping spatial duplicates within the same grid cell
df = df.drop_duplicates(subset=['lat_grid', 'lon_grid'], keep='first').copy()

print(f"Dataset after thinning (1km): {len(df)} records")

df[['decimalLatitude', 'lat_grid', 'decimalLongitude', 'lon_grid']].head()

We have reduced the dataset by nearly 90% by removing extremely close sightings. We are now left with 29,300 records to train our model.

In [ ]:
# Counting how many records exist per country and sorting from highest to lowest.
# These represent the specific grid cells where sightings have occurred.
country_counts = df['countryCode'].value_counts()
total_count = df

print("--- Sightings by Country in Europe ---")
print(country_counts)

## Getting presences and their geographical data


This section handles the integration with Google Earth Engine (GEE) by authenticating and initializing the API. The local dataset is streamlined to include only essential spatial variables (latitude, longitude, and year) and is then converted into an Earth Engine FeatureCollection. This transformation shifts the data from a local Python environment to the cloud, enabling a interactive mapping of Vespa velutina occurrences across Europe

In [ ]:
import sys
import geemap
import ee  # Import the Earth Engine library

# Install geojson library if not already installed
!pip install geojson

# Authenticate and Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='vespa-489513')

# Filtering only the required columns for the spatial model
required_columns = ['decimalLatitude', 'decimalLongitude', 'year']
df = df[required_columns].copy()

# Converting the pandas DataFrame to an Earth Engine FeatureCollection
# These will be our spatial points for all subsequent analysis
geospatial_points = geemap.pandas_to_ee(
    df,
    latitude='decimalLatitude',
    longitude='decimalLongitude'
)

"""
UNCOMMENT THIS IF YOU WANT TO SEE THE MAP OF PRESENCES


# Visualizing the points on an interactive map
Map = geemap.Map()
Map.addLayer(geospatial_points, {'color': 'red'}, 'Vespa velutina (Optimized)')
Map.centerObject(geospatial_points, 6)
Map
"""

--------------------------------------------

Given that the existing literature provides clear indicators of the environmental requirements for *Vespa velutina*, we will bypass **Principal Component Analysis (PCA)** in favor of a more interpretable approach. Instead, we will perform a **Multicollinearity Analysis** using the **Variance Inflation Factor (VIF)** to ensure our predictors are independent.
The candidate variables for the model include:

* **Bioclimatic (WorldClim):** Mean annual temperature (**bio01**), Max temperature of the warmest month (**bio05**), Min temperature of the coldest month (**bio06**), and Annual temperature range (**bio07**).
* **Precipitation (WorldClim):** Annual precipitation (**bio12**), Precipitation of the wettest month (**bio13**), Precipitation of the driest month (**bio14**), and Precipitation seasonality (**bio15**).
* **Topography (NASA SRTM):** Elevation and Slope.
* **Land Cover & Infrastructure:** ESA WorldCover (land type) and proximity to roads or urban areas (OpenStreetMap/GEE).

In [ ]:
# PREDICTOR CONFIGURATION

# Define the Area of Interest (AOI) with a 50km buffer
aoi = geospatial_points.geometry().bounds().buffer(50000)

# Bioclimatic variables (WorldClim)
bioclim = ee.Image('WORLDCLIM/V1/BIO').clip(aoi)

# Topography variables (SRTM)
srtm = ee.Image('USGS/SRTMGL1_003').clip(aoi)
elevation = srtm.select('elevation')
slope = ee.Terrain.slope(srtm).rename('slope')

# Land cover (ESA WorldCover)
land_cover = ee.ImageCollection("ESA/WorldCover/v100").first().clip(aoi).select(['Map'], ['landcover'])

# Infrastructure (Calculating distance to built-up areas/roads)
# Note: Using class 50 (Built-up) from WorldCover to estimate urban proximity
dist_roads = land_cover.eq(50).fastDistanceTransform().sqrt().multiply(1000).rename('dist_roads')

# Merging all bands into a single multi-dimensional image stack
predictors = bioclim.addBands(elevation).addBands(slope).addBands(land_cover).addBands(dist_roads).select([
    'bio01', 'bio05', 'bio06', 'bio07', 'bio12', 'bio13', 'bio14', 'bio15',
    'elevation', 'slope', 'landcover', 'dist_roads'
])

# SAMPLING
# Extracting environmental data for each presence point (Climate + Topography + Land Cover + Proximity)
sampled_presence_data = predictors.sampleRegions(
    collection=geospatial_points,
    properties=['year'],
    scale=1000,
    geometries=True
)

### MAP visualization of some useful environmental data (uncomment to see results)

In [ ]:

"""
predictors = bioclim.addBands(elevation).addBands(slope).addBands(land_cover).addBands(dist_roads).select([
    'bio01', 'bio05', 'bio06', 'bio07', 'bio12', 'bio13', 'bio14', 'bio15', 'elevation', 'slope', 'landcover', 'dist_roads'
])

# Temperature Palette (Bio01, Bio06, etc.) - From cold blue to heat red
vis_temp = {'min': -50, 'max': 300, 'palette': ['blue', 'cyan', 'green', 'yellow', 'red']}

# Precipitation Palette (Bio12, Bio14) - From dry white to rainfall blue
vis_precip = {'min': 0, 'max': 2500, 'palette': ['white', 'lightblue', 'blue', 'darkblue']}

# Elevation Palette
vis_el = {'min': 0, 'max': 2500, 'palette': ['006600', '002200', 'fff700', 'ab7634', 'c7d6ec', 'ffffff']}

# Slope Palette - From gentle green to steep red
vis_slope = {'min': 0, 'max': 45, 'palette': ['white', 'orange', 'red']}

# Official ESA WorldCover Palette (Landcover)
vis_lc = {'bands': ['landcover'], 'min': 10, 'max': 100,
          'palette': ['006400', 'ffbb22', 'ffff4c', 'f096ff', 'fa0000', 'b4b4b4', 'f0f0f0', '0064ff', '00bb88', 'ffff4c']}

# Distance to Roads Palette - From red (close) to white (far)
vis_dist = {'min': 0, 'max': 10000, 'palette': ['red', 'orange', 'white']}

Map = geemap.Map()
Map.add_basemap('HYBRID')

# Adding layers one by one (toggle them in the layer menu)
Map.addLayer(predictors.select('bio01'), vis_temp, 'Annual Mean Temperature (Bio01)', False)
Map.addLayer(predictors.select('bio06'), vis_temp, 'Min Temperature of Coldest Month (Bio06)', False)
Map.addLayer(predictors.select('bio12'), vis_precip, 'Annual Precipitation (Bio12)', False)
Map.addLayer(predictors.select('elevation'), vis_el, 'Elevation (SRTM)', False)
Map.addLayer(predictors.select('slope'), vis_slope, 'Slope')
Map.addLayer(predictors.select('landcover'), vis_lc, 'Land Cover (ESA WorldCover)')
Map.addLayer(predictors.select('dist_roads'), vis_dist, 'Proximity to Urban/Road Areas')

# Adding presence points as a visual reference
Map.addLayer(geospatial_points, {'color': 'magenta'}, 'Actual Nests (Points)')

# Center the map
Map.centerObject(geospatial_points, 7)
Map
"""



### VIF

In this section, we call the **VIF_variable_selector** script to perform a feature selection. This process identifies the variables that best represent our data from the set of all potentially useful predictors we previously considered. Once we know the golden set there is no need to execute this, but you can uncomment it to see the results!

In [ ]:
"""
from VIF_variable_selector import run_vif_cleaner

print("Downloading data from GEE...")
df_total = geemap.ee_to_df(sampled_presence_data)

# Setting the threshold to 10.0, though 5.0 could also be considered.
threshold = 10.0

print("Starting redundancy purge...")
final_variables, vif_report = run_vif_cleaner(df_total, threshold)

print("\n--- GOLDEN SUBSET ---")
print(f"Surviving variables: {final_variables}")
print("\nFinal VIF values:")
print(vif_report)
"""

This has been used to generate the graphs present in the documentation.
You can uncomment it too

In [ ]:
"""
import matplotlib.pyplot as plt

data = {
    'Variable': ['bio05', 'bio06', 'bio13', 'bio14', 'elevation', 'bio01', 'bio15', 'slope', 'bio07', 'bio12', 'dist_roads', 'landcover'],
    'VIF': [500, 104.91, 24.71, 10.16, 2.86, 2.36, 2.34, 1.80, 1.61, 1.35, 1.31, 1.11],
    'Subset': ['Dropped', 'Dropped', 'Dropped', 'Dropped', 'Golden', 'Golden', 'Golden', 'Golden', 'Golden', 'Golden', 'Golden', 'Golden']
}

df = pd.DataFrame(data)
# Invert the order so the "Golden" subset appears at the top of the horizontal chart
df = df.iloc[::-1]

# Red for Dropped, Blue for Golden
colors = ['#e74c3c' if s == 'Dropped' else '#3498db' for s in df['Subset']]

plt.figure(figsize=(10, 8))

# Create horizontal bar chart
bars = plt.barh(df['Variable'], df['VIF'], color=colors, edgecolor='black', alpha=0.8)

# Apply log scale to the X-axis for better visibility of low VIF values
plt.xscale('log')

# Add VIF value labels to the end of each bar
for bar in bars:
    xval = bar.get_width()
    plt.text(xval * 1.1, bar.get_y() + bar.get_height()/2,
             f'{xval if xval < 400 else "inf"}',
             va='center', ha='left', fontsize=10, fontweight='bold')

# Add vertical line at the VIF=10 threshold
plt.axvline(x=10, color='gray', linestyle='--', alpha=0.6, label='Threshold (VIF=10)')

plt.title('VIF Analysis: Feature Selection for Vespa velutina Model', fontsize=14, pad=20)
plt.xlabel('VIF Value (Log Scale)')
plt.ylabel('Environmental Predictors')

plt.grid(axis='x', linestyle=':', alpha=0.5)

# Custom legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color='#3498db', lw=4, label='Golden Subset (Accepted)'),
                   Line2D([0], [0], color='#e74c3c', lw=4, label='Redundant (Dropped)')]
plt.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

"""

### Feature Filtering: Final Presence Selection

In this section, we filter the presence records to retain only the specific environmental variables of interest for our model based on their VIF

In [ ]:
# Variables that maintained a VIF < 10 (FINAL PRESENCE COLLECTION)
golden_variables = [
    'elevation', 'bio01', 'bio15', 'slope',
    'bio07', 'bio12', 'dist_roads', 'landcover'
] # dist_roads sometimes causes bugs; we can re-insert it later if needed

# Filter the multi-band image to keep only the selected predictors
final_predictors = predictors.select(golden_variables)

# Sampling the final environmental data at presence locations
final_presence_data = final_predictors.sampleRegions(
    collection=geospatial_points,
    properties=['year'],
    scale=1000,
    geometries=True
)

# Labeling these records as class 1 (Presence)
final_presence_data = final_presence_data.map(lambda f: f.set('class', 1))

----------------------------------------------------------------

# Generating pseudoabsences

Esta sección es para enseñarle al modelo dónde NO hay avispas, para ello generamos pseudoausencias. En un principio tenía pensado hacerlas random, pero rápidamente surjen varios problemas:

1- qué pasa si en la zona de avistamiento hay un punto de ausencia cuando en realidad debería de ser de presencia? A lo mejor ese lugar es un bosque profundo y no pasan personas para avistar avispas. O igual el punto random cae en un sitio en el que sí ha habido avistamientos. Hacerlo random, sobre todo teniendo en cuenta que lo hacemos en un rango aoi donde sí se han producido avistamientos, no es una buena idea

2- Si efectivamente el punto random cae en un sitio donde efectivamente la avispa está ausente, ¿cómo sabemos que está ausente porque las condiciones no son las adecuadas? a lo mejor simplemente la avispa todavía no ha llegado a esa zona, o hay una barrera geográfica que le impide llegar aunque las condiciones sí sean las adecuadas en la zona en sí (imaginemos el paraíso de las avispas rodeado de unas montañas inexpugnables, la avispa no llegará al paraíso pero no porque el paraiso no sea bueno, si no porque no tiene forma de llegar ahí).

Haremos un híbrido y luego lo justificaremos en el documento y en el vídeo.

## SM1

Estrategia para generar pseudoausencias 1: SM1 random
Generamos puntos random en nuestra zona de interés que no coincidan con los puntos de presencia. Una vez generados los visualizamos y guardamos en un dataset para poder compararlos en un futuro y no tener que ejecutar esta sección cada vez.

In [ ]:

#GENERACIÓN DE AUSENCIAS SM1 (IMPORTANTE CHEKIARLO PARA VER MI COLECCIÓN DE PSEUDOAUSENCIAS)

aoi = puntos_geospatiales.geometry().bounds().buffer(2000000) #aoi devuelve una figura geométrica de 2000km alrededor de los puntos

#Creamos la imagen de los puntos de presencia
presencia_img = ee.Image().paint(puntos_geospatiales, 1000)

# Creamos la máscara para evitar el oceano
landcover = ee.ImageCollection("ESA/WorldCover/v100").first()
mascara_tierra = landcover.neq(80)

# Creamos una imagen de valor 1 en toda la AOI y "borramos" donde hay presencias
# Usamos un buffer de 1000m (1km) para asegurar que no caigan en el mismo píxel por si acaso

mascara_exclusion = presencia_img.fastDistanceTransform().sqrt().gt(1)


area_muestreo = ee.Image(1).clip(aoi).updateMask(mascara_exclusion).updateMask(mascara_tierra) #Area muestreo es el área permitida donde puede haber ausencias

# Generamos las 29.306 pseudo-ausencias (Seguimos un ratio 1:1), aunque se pueden generar algunos menos
# si el punto random cae justo en una presencia y se descarta.

pseudo_ausencias_raw = predictores_final.updateMask(area_muestreo).sample(
    region=aoi,
    scale=1000,
    numPixels=1000000, #Esto se trunca con el raw_limit
    seed=67,
    geometries=True
)

# Etiquetamos y unimos el dataset
# Clase 1 = Presencia, Clase 0 = Pseudo-ausencia
# 2. Etiquetamos las ausencias como Clase 0
# Importante: Estas ya traen las variables de 'predictores_final' por el paso anterior
ausencias = pseudo_ausencias_raw.limit(28887).map(lambda f: f.set('class', 0))

# 3. Usamos las presencias que YA muestreamos antes (las que tienen variables_oro)
# Asegúrate de que este objeto existe en tu código (lo creaste en la celda anterior)
presencias_final = presencias_final.map(lambda f: f.set('class', 1))

# 4. Unimos los dos datasets "completos"
dataset_total_SM1 = presencias_final.merge(ausencias)

"""
print(f"Dataset SM1 listo. AUSENCIAS: {ausencias.size().getInfo()}, deberían de ser cerca de 29306)")
print(f"Dataset SM1 listo. PRESENCIAS:: {presencias_final.size().getInfo()}, deberían de ser cerca de 29306)")
#print(f"Dataset SM1 listo. Total puntos: {dataset_total_SM1.size().getInfo()}, deberían de ser cerca de 58.512")
"""

In [ ]:

# SECCIÓN DE VISUALIZACIÓN SM1
Map = geemap.Map()
Map.add_basemap('HYBRID')

estilo_vespa = {'color': 'red', 'pointSize': 4, 'pointShape': 'circle', 'width': 1}
estilo_vacio = {'color': '00ffff', 'pointSize': 4, 'pointShape': 'square', 'width': 1} # Cian más grande

# Metemos una opción de máscara para que se vea en qué región se pueden colocar los puntos
vis_mask = {
    'min': 0,
    'max': 1,
    'palette': ['000000', 'yellow'], # Changed 'transparent' to '000000' (black)
    'opacity': 0.35 # Menos opaco para ver el terreno
}
Map.addLayer(area_muestreo, vis_mask, 'Debug: Área de Muestreo (Permitida)')
Map.addLayer(ausencias.style(**estilo_vacio), {}, 'Pseudo-ausencias (Clase 0)')
Map.addLayer(presencias.style(**estilo_vespa), {}, 'Presencias Vespa (Clase 1)')

Map.centerObject(puntos_geospatiales, 6)
Map


In [ ]:
import geemap

# SCRIPT DE DEBUG Y VISUALIZACIÓN PARA PUNTOS SM1 CON DATOS

# 1. Separar para verificar (esto es lo que pedías)
puntos_presencia = dataset_total_SM1.filter(ee.Filter.eq('class', 1))
puntos_ausencia = dataset_total_SM1.filter(ee.Filter.eq('class', 0))

# 2. Imprimir diagnóstico en la consola
print(f"✅ Presencias finales: {puntos_presencia.size().getInfo()}")
print(f"✅ Ausencias finales: {puntos_ausencia.size().getInfo()}")

# 3. COMPROBACIÓN CRÍTICA DE COLUMNAS
# Esto nos dirá si ambos grupos tienen las mismas variables_oro
print("Columnas en Presencia:", presencias_final.first().propertyNames().getInfo())
print("Columnas en Ausencia:", puntos_ausencia.first().propertyNames().getInfo())

# 4. Visualización en el Mapa
Map = geemap.Map()

# --- ESTA ES LA LÍNEA QUE EVITA EL ERROR 403 ---
Map.add_basemap('HYBRID') # Usa satélite de Google en lugar de OpenStreetMap
# -----------------------------------------------

# Dibujamos las ausencias en ROJO (pequeñas para no tapar todo)
Map.addLayer(puntos_ausencia,
             {'color': 'red', 'pointSize': 1},
             'Pseudo-ausencias (Clase 0)')

# Dibujamos las presencias en AZUL (más grandes para que resalten)
Map.addLayer(puntos_presencia,
             {'color': 'blue', 'pointSize': 3},
             'Presencias Reales (Clase 1)')

Map.centerObject(puntos_geospatiales, 5)
Map

Exportamos las pseudoausencias generadas con SM1 para no tener que hacer el proceso de limpieza cada vez y luego poder comparar los modelos utilizando diferentes técnicas.
Esto lo mantengo comentado por si acaso tengo que volver a hacerlo.



In [ ]:

# EXTRACCIÓN Y GUARDADO DE COORDENADAS (APLICABLE PARA TODAS CAMBIANDO UN PAR DE COSILLAS)

def extraer_posicion(feature):
    coords = feature.geometry().coordinates()
    return feature.set({
        'latitude': coords.get(1),
        'longitude': coords.get(0)
    })


ausencias_finales = ausencias.map(extraer_posicion).select(['latitude', 'longitude', 'class'])
nombre_csv = 'SM1_absences_COORDENADASDef.csv'
geemap.ee_to_csv(ausencias_finales, filename=nombre_csv)


## SM2

Estrategia para generar pseudoausencias 2: SM2 límite geográfico
Generamos puntos random en nuestra zona de interés que no coincidan con los puntos de presencia, pero SOLO en un área suficientemente cercana a nuestros puntos para obligar al modelo a aprender de zonas que estén relativamente cerca de nuestra área. No tiene tanto impacto porque se solapa bastante con aoi. Una vez generados los visualizamos y guardamos en un dataset para poder compararlos en un futuro y no tener que ejecutar esta sección cada vez.

In [ ]:
# GENERACIÓN DE AUSENCIAS SM2 (SPATIALLY CONSTRAINED)

# Definimos el AOI como una "nube" de buffers de 350km alrededor de los nidos
# Quitamos.bounds() para que no sea un rectángulo, sino el área real de influencia

aoi_sm2 = puntos_geospatiales.geometry().buffer(350000)

presencia_img = ee.Image().paint(puntos_geospatiales, 1)

landcover = ee.ImageCollection("ESA/WorldCover/v100").first()
mascara_tierra = landcover.neq(80)

mascara_exclusion = presencia_img.fastDistanceTransform().sqrt().gt(1)

area_muestreo_sm2 = ee.Image(1).clip(aoi_sm2).updateMask(mascara_exclusion).updateMask(mascara_tierra)

pseudo_ausencias_sm2_raw = area_muestreo_sm2.sample(
    region=aoi_sm2,
    scale=1000,
    numPixels=60000, #Esto se trunca a 29306
    seed=67,
    geometries=True
)

pseudo_ausencias_sm2 = pseudo_ausencias_sm2_raw.limit(29306)

# Etiquetamos y unimos el dataset
presencias = puntos_geospatiales.map(lambda f: f.set('class', 1))
ausencias_sm2 = pseudo_ausencias_sm2.map(lambda f: f.set('class', 0))

# El dataset final para el modelo SM2
dataset_total_sm2 = presencias.merge(ausencias_sm2)

print(f"Dataset SM2 listo con: {dataset_total_sm2.size().getInfo()} puntos.")
print("La IA ahora solo aprenderá de ausencias situadas a menos de 350km de los nidos.")

In [ ]:
# SECCIÓN DE VISUALIZACIÓN SM2

Map = geemap.Map()
Map.add_basemap('HYBRID')

# Estilos definidos para Vespa y Ausencias
estilo_vespa = {'color': 'red', 'pointSize': 4, 'pointShape': 'circle', 'width': 1}
estilo_vacio = {'color': '00ffff', 'pointSize': 4, 'pointShape': 'square', 'width': 1}

# Configuración de la máscara de búsqueda (Nube de 350km)
# Usamos unmask(0) para que los "huecos" de 1km se vean negros
# El clip(aoi_sm2) asegura que solo veamos la forma de la nube y no un rectángulo
vis_mask = {
    'min': 0,
    'max': 1,
    'palette': ['000000', 'yellow'], # Negro para prohibido, Amarillo para permitido
    'opacity': 0.45 # Ajustado para que sea vistoso pero deje ver el mapa
}

# Representamos la zona de muestreo permitida para SM2
Map.addLayer(area_muestreo_sm2.unmask(0).clip(aoi_sm2), vis_mask, 'Región de Muestreo SM2 (350km)')

# Añadir los puntos (rasterizados para fluidez)
Map.addLayer(ausencias_sm2.style(**estilo_vacio), {}, 'Pseudo-ausencias SM2 (Clase 0)')
Map.addLayer(presencias.style(**estilo_vespa), {}, 'Presencias Vespa (Clase 1)')

# Centrar el mapa en la invasión
Map.centerObject(puntos_geospatiales, 6)
Map

In [ ]:
"""
# EXTRACCIÓN Y GUARDADO DE COORDENADAS (APLICABLE PARA TODAS CAMBIANDO UN PAR DE COSILLAS)

def extraer_posicion(feature):
    coords = feature.geometry().coordinates()
    return feature.set({
        'latitude': coords.get(1),
        'longitude': coords.get(0)
    })


ausencias_finales = ausencias_sm2.map(extraer_posicion).select(['latitude', 'longitude', 'class'])
nombre_csv = 'SM2_absences_COORDENADASDef.csv'
geemap.ee_to_csv(ausencias_finales, filename=nombre_csv)"""

SM3: Cambiamos de clima para forzar a que el modelo se ajuste

# EN ESTA SECCIÓN DE AQUÍ ABAJO VAMOS A ENTRENAR EL RANDOM FOREST CON LOS MODELOS DE PRESENCIA Y CON LAS PSEUDOAUSENCIAS DEL SM1



In [ ]:
import geemap

# Definimos una paleta de colores: de blanco/amarillo (poco probable) a rojo oscuro (muy probable)
vis_params = {
    'min': 0,
    'max': 1,
    'palette': ['#ffffff', '#fdfd96', '#ffb347', '#ff6961', '#8b0000']
}

Map = geemap.Map()
Map.add_basemap('HYBRID')

# Añadimos el mapa de fitness
# Nota: Si quieres ver probabilidades (0 a 1) en lugar de solo 0 o 1,
# asegúrate de que el clasificador esté en modo 'PROBABILITY'
prob_classifier = classifier.setOutputMode('PROBABILITY')
mapa_probabilidades = predictores_final.select(nombres_variables).classify(prob_classifier)

Map.addLayer(mapa_probabilidades, vis_params, 'Índice de Fitness (Probabilidad)')

# Añadimos tus presencias originales en azul para comparar
Map.addLayer(presencias_final, {'color': 'blue'}, 'Presencias Reales')

Map.centerObject(puntos_geospatiales, 5)
Map

In [ ]:
# 1. Definimos los predictores GLOBALES (sin recortes)
bioclim_global = ee.Image('WORLDCLIM/V1/BIO')
srtm_global = ee.Image('USGS/SRTMGL1_003')
# El landcover global es necesario para que dist_roads funcione en cualquier sitio
landcover_global = ee.ImageCollection("ESA/WorldCover/v100").first()

# 2. Calculamos las capas derivadas a nivel global
elev_global = srtm_global.select('elevation')
slope_global = ee.Terrain.slope(srtm_global).rename('slope')
# Calculamos la distancia a zonas urbanas (valor 50) en todo el mapa
dist_roads_global = landcover_global.eq(50).fastDistanceTransform().sqrt().multiply(1000).rename('dist_roads')
terreno_global = landcover_global.select(['Map'], ['landcover'])

# 3. Juntamos las bandas (Deben tener los mismos nombres que usaste al entrenar)
predictores_global = bioclim_global.addBands(elev_global) \
                                   .addBands(slope_global) \
                                   .addBands(terreno_global) \
                                   .addBands(dist_roads_global)

# 4. APLICAMOS EL MODELO YA ENTRENADO
# Usamos .classify() sobre la imagen global.
# El 'classifier' ya tiene el conocimiento guardado.
mapa_fitness_final = predictores_global.select(nombres_variables).classify(classifier.setOutputMode('PROBABILITY'))

# 5. Mostrar en el mapa
Map = geemap.Map()
Map.add_basemap('HYBRID')
Map.addLayer(mapa_fitness_final, {'min': 0, 'max': 1, 'palette': ['white', 'yellow', 'red']}, 'Fitness Vespa Global')
Map

## REVISIONES

Cosas a arreglar o revisar:

5- ENTRENAR EL MODELO, LUEGO HACER VALIDACIÓN CRUZADA (SE VERÁ COMO SE HACE) PARA VER CÓMO DE BUENO ES.

6- REVISAR, DOCUMENTAR, README DE GITHUB


Enlaces útiles:

https://ramirodcrego.com/teaching/gee/

https://developers.google.com/earth-engine/tutorials/community/species-distribution-modeling?hl=es-419

https://www.tandfonline.com/doi/full/10.1080/10095020.2025.2546507#abstract

https://www.researchgate.net/publication/284246225_A_multi-scale_approach_to_identify_invasion_drivers_and_invaders'_future_dynamics

https://onlinelibrary.wiley.com/doi/full/10.1002/ece3.70029

https://besjournals.onlinelibrary.wiley.com/doi/10.1111/j.2041-210X.2011.00172.x

https://biogeography-usc.org/positive/Prediction.html

https://www.sciencedirect.com/science/article/abs/pii/S0006320711001315

https://revistaecosistemas.net/index.php/ecosistemas/article/view/2987/1962

https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0071218 (PAPER SM4)

https://gidahatari.com/ih-es/bioclim-un-sistema-de-analisis-y-prediccion-de-bioclimas (BIOCLIM)

https://www.researchgate.net/publication/226562656_DOMAIN_a_flexible_modelling_procedure_for_mapping_potential_distributions_of_plants_and_animals (DOMAIN)

https://www.sciencedirect.com/science/article/pii/S0304380011000780 (PAPER 1 JUSTIFICACIÓN BUFFER)

https://besjournals.onlinelibrary.wiley.com/doi/10.1111/j.2041-210X.2010.00036.x (PAPER 2 JUSTIFICACIÓN BUFFER)

https://besjournals.onlinelibrary.wiley.com/doi/full/10.1111/1365-2664.12724 (PAPER 3 JUSTIFICACIÓN BUFFER)

https://www.plant-animal.es/pdfs/Herrera.2024.Ecosistemas.pdf (paper justificación presencias)